# 🏀 バスケ試合動画の選手を「色分け＆追跡」する（YOLO + SAM2）

note記事「[バスケの試合動画をAIに見せたら、選手が勝手に色分け＆追跡された](https://note.com/hirosuke_0520/n/n85d57196fffd)」の内容を
**Google Colab の無料GPU** で再現するノートブックです。

このノートブックは **1試合分の長い動画でもメモリ落ち（OOM）しない逐次処理** に対応しています。

**処理の流れ**

1. **パス1 … 検出＆追跡**: YOLO11 で選手を検出、ByteTrack で各選手にIDを付与（この段階では画像を溜め込まず、検出データだけ収集）
2. **チーム分け**: 各選手のユニフォーム色（LAB色空間）でKMeansクラスタリング。ベンチ/観客はサイズと動き量で除外
3. **パス2 … 描画**: 動画をディスクから1フレームずつ読み直し、チーム色の枠を描いて書き出し（メモリ一定）
4. **CSV出力**: フレームごとの選手位置を `detections.csv` に保存（→ 個人成績・戦術分析の土台）
5. **（任意）SAM2 で個人の精密追跡**

> 💡 **使い方**: 「ランタイム → ランタイムのタイプを変更 → **GPU (T4)**」にしてから、上のセルから順に実行してください。


## 1. 環境セットアップ

In [ ]:
!nvidia-smi -L || echo "⚠️ GPUが見つかりません。ランタイム → ランタイムのタイプを変更 → GPU を選択してください。"


In [ ]:
%pip install -q "ultralytics>=8.3.0" scikit-learn
print("✅ インストール完了")


In [ ]:
import cv2, csv, math
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else "cpu"


## 2. 解析する動画を用意する

- **A. 自分の動画をアップロード**: 下のセルをそのまま実行してファイルを選択
- **B. サンプルで試す**: `USE_SAMPLE = True`


In [ ]:
USE_SAMPLE = False  # サンプル動画を使う場合は True

VIDEO_PATH = None
if USE_SAMPLE:
    import urllib.request
    sample_url = "https://media.roboflow.com/supervision/video-examples/basketball-1.mp4"
    VIDEO_PATH = "input.mp4"
    try:
        urllib.request.urlretrieve(sample_url, VIDEO_PATH)
        print("✅ サンプル動画を取得:", VIDEO_PATH)
    except Exception as e:
        print("⚠️ サンプル取得に失敗。手動アップロードに切替:", e); USE_SAMPLE = False

if not USE_SAMPLE:
    from google.colab import files
    print("動画ファイル(mp4)を選択してください…")
    uploaded = files.upload()
    VIDEO_PATH = list(uploaded.keys())[0]
    print("✅ アップロード:", VIDEO_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS)
N_TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FRAME_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); FRAME_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"📹 {VIDEO_PATH}: {FRAME_W}x{FRAME_H}, {FPS:.1f}fps, {N_TOTAL}フレーム (~{N_TOTAL/max(FPS,1):.1f}秒)")


### （任意）動画を短く切ってから試す

**1試合分をそのまま流してもメモリは大丈夫**な作りですが、初回の動作確認は短い方が速いです。
`TRIM_SECONDS` に秒数を入れると先頭だけ切り出します。**全部やるときは `None` のままでOK**。


In [ ]:
TRIM_SECONDS = None  # 例: 30 で先頭30秒。None なら動画全体

if TRIM_SECONDS:
    trimmed = "trimmed.mp4"
    !ffmpeg -y -i "{VIDEO_PATH}" -t {TRIM_SECONDS} -c:v libx264 -an "{trimmed}" -loglevel error
    VIDEO_PATH = trimmed
    cap = cv2.VideoCapture(VIDEO_PATH)
    FPS = cap.get(cv2.CAP_PROP_FPS)
    N_TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    FRAME_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); FRAME_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f"✂️ 先頭{TRIM_SECONDS}秒に切出し → {VIDEO_PATH} ({N_TOTAL}フレーム)")
else:
    print("トリミングなし（動画全体を使用）")


## 3. パス1: 選手の検出＆追跡（YOLO11 + ByteTrack）

動画をストリーム処理し、**各フレームの検出（フレーム番号・ID・bbox）** と **各IDのユニフォーム色サンプル** だけを集めます。
画像そのものは保持しないので、長い動画でもメモリは増えません。


In [ ]:
from ultralytics import YOLO

# 長い動画で速度が欲しければ "yolo11n.pt"、精度重視なら "yolo11x.pt"
det_model = YOLO("yolo11m.pt")
PERSON_CLASS = 0
CONF = 0.35

track_boxes  = defaultdict(list)  # tid -> [(frame, x1,y1,x2,y2), ...]
track_colors = defaultdict(list)  # tid -> [LAB色サンプル, ...]（最大MAX_COLOR件）
MAX_COLOR = 60

def torso_lab(frame, box):
    """bboxの上半身中央（ユニフォーム）の代表色をLABで返す"""
    x1, y1, x2, y2 = map(int, box)
    bw, bh = x2 - x1, y2 - y1
    cx1, cx2 = x1 + int(bw * 0.28), x2 - int(bw * 0.28)
    cy1, cy2 = y1 + int(bh * 0.18), y1 + int(bh * 0.45)
    crop = frame[max(cy1, 0):cy2, max(cx1, 0):cx2]
    if crop.size == 0:
        return None
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    return np.median(lab, axis=0)  # [L, a, b]

results = det_model.track(
    source=VIDEO_PATH, stream=True, persist=True,
    classes=[PERSON_CLASS], conf=CONF, tracker="bytetrack.yaml",
    device=DEVICE, verbose=False,
)

n_seen = 0
for f_idx, r in enumerate(results):
    n_seen += 1
    if r.boxes is None or r.boxes.id is None:
        continue
    frame = r.orig_img
    boxes = r.boxes.xyxy.cpu().numpy()
    ids = r.boxes.id.cpu().numpy().astype(int)
    for box, tid in zip(boxes, ids):
        track_boxes[tid].append((f_idx, *[float(v) for v in box]))
        if len(track_colors[tid]) < MAX_COLOR:
            c = torso_lab(frame, box)
            if c is not None:
                track_colors[tid].append(c)
    if f_idx % 300 == 0:
        print(f"  ...{f_idx}/{N_TOTAL} フレーム処理")

print(f"✅ パス1完了: {n_seen}フレーム / 検出IDユニーク数 = {len(track_boxes)}")


## 4. 選手フィルタ ＋ チーム分け

- **ベンチ・観客・審判の除外**: ①出現フレーム数が少ない ②枠が小さい（遠い/座っている）③ほとんど動かない、を外れ値として除外
- **チーム分け**: 各選手の代表ユニフォーム色（LAB中央値）で KMeans(2)。LABの明度Lが効くので「白 vs 紺」のような明暗差に強い

しきい値（`MIN_FRAMES` / `MIN_HEIGHT_RATIO` / `MIN_MOVEMENT`）は映像に合わせて調整できます。


In [ ]:
from sklearn.cluster import KMeans

# --- フィルタのしきい値（映像に合わせて調整）---
MIN_FRAMES       = max(5, N_TOTAL // 30)   # これ未満しか映らないIDは除外
MIN_HEIGHT_RATIO = 0.06                     # bbox高さが画面高さのこの割合未満は除外（遠い/ベンチ）
MIN_MOVEMENT     = 25.0                      # 総移動量(px)がこれ未満は除外（座ってる人）

def track_stats(dets):
    arr = np.array([[x1, y1, x2, y2] for (_, x1, y1, x2, y2) in dets])
    heights = arr[:, 3] - arr[:, 1]
    cx = (arr[:, 0] + arr[:, 2]) / 2.0
    cy = (arr[:, 1] + arr[:, 3]) / 2.0
    movement = float(np.sum(np.hypot(np.diff(cx), np.diff(cy)))) if len(cx) > 1 else 0.0
    return float(np.median(heights)), movement

valid_ids, feats = [], []
report = []
for tid, dets in track_boxes.items():
    n = len(dets)
    med_h, movement = track_stats(dets)
    ok = (n >= MIN_FRAMES and med_h >= MIN_HEIGHT_RATIO * FRAME_H
          and movement >= MIN_MOVEMENT and len(track_colors[tid]) >= 3)
    report.append((tid, n, round(med_h, 1), round(movement, 1), ok))
    if ok:
        valid_ids.append(tid)
        feats.append(np.median(np.array(track_colors[tid]), axis=0))  # [L,a,b]

print(f"選手候補: {len(valid_ids)} / 全ID {len(track_boxes)} （除外 {len(track_boxes)-len(valid_ids)}）")

team_of = {}
if len(valid_ids) >= 2:
    X = np.array(feats)
    km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
    for tid, lab in zip(valid_ids, km.labels_):
        team_of[tid] = int(lab)
    # どちらのチームが明るい(白)側か: 平均L
    mean_L = [X[km.labels_ == t][:, 0].mean() for t in (0, 1)]
    bright_team = int(np.argmax(mean_L))
    print(f"✅ チーム分け完了  (チーム{bright_team}が明るい色側)")
    for t in (0, 1):
        members = [i for i in valid_ids if team_of[i] == t]
        print(f"  チーム{t}: {len(members)}人  IDs={members}")
else:
    print("⚠️ 選手が2人以上検出できませんでした。CONFやフィルタ値を調整してください。")

TEAM_COLORS = {0: (0, 0, 255), 1: (255, 128, 0)}  # 赤 / 青(BGR)


### （確認用）フィルタで何が除外されたか

`ok=False` が除外されたID。ベンチ・観客が落とせているか確認し、
本来の選手まで消えていたらしきい値を緩めてください。


In [ ]:
print(f"{'ID':>4} {'frames':>7} {'height':>7} {'move':>7}  keep")
for tid, n, h, mv, ok in sorted(report, key=lambda r: -r[1]):
    print(f"{tid:>4} {n:>7} {h:>7} {mv:>7}  {'✓' if ok else '×'}")


## 5. パス2: 色分け動画の書き出し ＋ CSV出力

動画をディスクから1フレームずつ読み直し、チーム色の枠を描いて書き出します（メモリ一定）。
同時に、各フレームの選手位置を `detections.csv` に保存します（個人成績・戦術分析の土台）。


In [ ]:
# フレーム番号 -> [(tid,x1,y1,x2,y2), ...]（選手として有効なIDのみ）
by_frame = defaultdict(list)
DRAW_NONPLAYERS = False  # True にすると除外IDもグレーで描画
for tid, dets in track_boxes.items():
    if tid not in team_of and not DRAW_NONPLAYERS:
        continue
    for (f_idx, x1, y1, x2, y2) in dets:
        by_frame[f_idx].append((tid, x1, y1, x2, y2))

out_path = "output_team_colored.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (FRAME_W, FRAME_H))
csv_file = open("detections.csv", "w", newline="")
cw = csv.writer(csv_file)
cw.writerow(["frame", "time_s", "track_id", "team", "x1", "y1", "x2", "y2", "cx", "foot_y"])

UNKNOWN = (160, 160, 160)
f_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    for (tid, x1, y1, x2, y2) in by_frame.get(f_idx, []):
        team = team_of.get(tid)
        color = TEAM_COLORS.get(team, UNKNOWN)
        p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
        cv2.rectangle(frame, p1, p2, color, 2)
        label = f"ID{tid}" + (f" T{team}" if team is not None else "")
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame, (p1[0], p1[1] - th - 6), (p1[0] + tw + 4, p1[1]), color, -1)
        cv2.putText(frame, label, (p1[0] + 2, p1[1] - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        cx = (x1 + x2) / 2.0
        cw.writerow([f_idx, round(f_idx / FPS, 3), tid, team,
                     int(x1), int(y1), int(x2), int(y2), round(cx, 1), int(y2)])
    writer.write(frame)
    f_idx += 1

cap.release(); writer.release(); csv_file.close()
print(f"✅ 書き出し完了: {out_path} / detections.csv ({f_idx}フレーム)")


In [ ]:
# 各選手の要約CSV（出場フレーム数・チーム）
with open("player_tracks.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["track_id", "team", "frames", "seconds"])
    for tid in valid_ids:
        w.writerow([tid, team_of.get(tid), len(track_boxes[tid]),
                    round(len(track_boxes[tid]) / FPS, 1)])
print("✅ player_tracks.csv を書き出しました")


In [ ]:
# Colab上でプレビュー（長い動画は先頭60秒だけ変換して再生）
!ffmpeg -y -t 60 -i output_team_colored.mp4 -vcodec libx264 -movflags +faststart preview.mp4 -loglevel error
from IPython.display import HTML
from base64 import b64encode
data_url = "data:video/mp4;base64," + b64encode(open("preview.mp4", "rb").read()).decode()
HTML(f'<video width="640" controls><source src="{data_url}" type="video/mp4"></video>')


## 6. 結果をダウンロード

In [ ]:
from google.colab import files
for f in ["output_team_colored.mp4", "detections.csv", "player_tracks.csv"]:
    if Path(f).exists():
        files.download(f)


## 7. 背番号OCR（選手の個人特定・検証）

トラッキングIDは途中で分裂します（同じ選手に別IDが振られる＝IDスイッチ）。
これを解決する王道が **背番号で同一選手を再結合（re-ID）** することです。

このセクションは **「この映像で背番号が読めるか」の検証** です。各選手IDについて複数フレームから
背番号の領域を切り出してOCRし、多数決で番号を推定 → **読み取り成功率**と「番号→IDの束ね」を出します。

> 💡 背番号OCRは小さく歪んだ数字が相手なので、体育館の引き画だと**読めないことも多い**です。
> ここでまず精度を測り、実用になりそうか判断してから個人成績に進みます。


In [ ]:
%pip install -q easyocr
import easyocr
from collections import Counter

# GPUで初回はモデルを自動ダウンロード
reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())
print("✅ EasyOCR 準備完了")


In [ ]:
# 各選手IDから背番号領域をサンプリングしてOCR
SAMPLES_PER_ID = 15          # 1IDあたり何フレーム試すか
NUM_REGION = dict(x=0.18, y=0.12, w=0.64, h=0.48)  # bbox内の背番号領域(胴体中央)の割合
MIN_CONF   = 0.30            # OCR信頼度の下限

# サンプリング計画: frame -> [(tid, box), ...]（1パスで読むためフレーム順にまとめる）
plan = defaultdict(list)
for tid in valid_ids:
    dets = sorted(track_boxes[tid])
    if not dets:
        continue
    idxs = np.linspace(0, len(dets) - 1, min(SAMPLES_PER_ID, len(dets))).astype(int)
    for k in idxs:
        f, x1, y1, x2, y2 = dets[k]
        plan[f].append((tid, (x1, y1, x2, y2)))

reads = defaultdict(list)  # tid -> [読めた数字文字列, ...]
targets = set(plan.keys())
cap = cv2.VideoCapture(VIDEO_PATH)
f_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if f_idx in targets:
        for tid, (x1, y1, x2, y2) in plan[f_idx]:
            x1, y1, x2, y2 = map(int, (x1, y1, x2, y2))
            bw, bh = x2 - x1, y2 - y1
            rx1 = x1 + int(bw * NUM_REGION["x"]); rx2 = rx1 + int(bw * NUM_REGION["w"])
            ry1 = y1 + int(bh * NUM_REGION["y"]); ry2 = ry1 + int(bh * NUM_REGION["h"])
            crop = frame[max(ry1, 0):ry2, max(rx1, 0):rx2]
            if crop.size == 0:
                continue
            crop = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)  # 拡大
            for (_, txt, conf) in reader.readtext(crop, allowlist="0123456789", detail=1):
                txt = txt.strip()
                if txt.isdigit() and 0 < len(txt) <= 2 and conf >= MIN_CONF:
                    reads[tid].append(txt)
    f_idx += 1
cap.release()
print(f"✅ OCR完了: {len(targets)}フレームをサンプリング / 読めたID数 = {len(reads)}")


In [ ]:
# 集計: 各IDの背番号を多数決で推定し、成功率を出す
AGREE_MIN = 0.5   # 多数決の一致率がこれ以上なら「確定」とみなす
VOTES_MIN = 2     # 最低票数

id_to_number, confident = {}, 0
print(f"{'ID':>4} {'team':>4} {'背番号':>5} {'票':>4} {'試行':>4} {'一致率':>6}")
for tid in valid_ids:
    c = Counter(reads.get(tid, []))
    if c:
        num, votes = c.most_common(1)[0]
        total = sum(c.values())
        agree = votes / total
        ok = (agree >= AGREE_MIN and votes >= VOTES_MIN)
        confident += ok
        id_to_number[tid] = num if ok else None
        mark = "✓" if ok else " "
        print(f"{tid:>4} {str(team_of.get(tid)):>4} {num:>5} {votes:>4} {total:>4} {agree:>6.2f} {mark}")
    else:
        id_to_number[tid] = None
        print(f"{tid:>4} {str(team_of.get(tid)):>4} {'?':>5} {0:>4} {0:>4} {'-':>6}")

rate = confident / max(len(valid_ids), 1)
print(f"\n📊 背番号が確定できたID: {confident}/{len(valid_ids)}  (成功率 {rate:.0%})")


In [ ]:
# 「番号 → 束ねられたID」= re-ID の効果。同じ番号の複数IDは同一選手の可能性が高い
by_number = defaultdict(list)
for tid, num in id_to_number.items():
    if num is not None:
        by_number[num].append(tid)

print("背番号ごとにまとめた選手（分裂IDの再結合の候補）:")
for num in sorted(by_number, key=lambda n: int(n)):
    ids = by_number[num]
    teams = {team_of.get(i) for i in ids}
    total_sec = round(sum(len(track_boxes[i]) for i in ids) / FPS, 1)
    print(f"  背番号 {num:>2}: IDs={ids}  team={teams}  合計出場 {total_sec}s")
print("\n→ ここが分裂なく1番号=数IDに収まっていれば、個人成績の集計に進めます。")


## 8. トラック統合（re-ID）: 分裂したIDを1人にまとめる

背番号OCRが弱くても効く方法です。**動きの連続性**で分裂IDをつなぎます:
「あるIDが途切れた直後・近い位置・同じチームで別IDが始まった」なら**同一選手**として結合。

これで「49個の断片 → 実際の約10人」に近づけ、匿名の選手IDごとに出場時間などを集計できます。
`MAX_GAP_S`（途切れ許容秒）と `MAX_SPEED`（許容移動速度）で挙動を調整できます。


In [ ]:
# 各トラックの開始/終了フレームと、その時の足元位置(cx, foot_y)を要約
summary = {}
for tid in valid_ids:
    ds = sorted(track_boxes[tid])
    f0, x1, y1, x2, y2 = ds[0];  c0 = ((x1 + x2) / 2.0, y2)
    f1, X1, Y1, X2, Y2 = ds[-1]; c1 = ((X1 + X2) / 2.0, Y2)
    summary[tid] = dict(f0=f0, f1=f1, c0=c0, c1=c1, team=team_of[tid], n=len(ds))

# --- 連結のしきい値 ---
MAX_GAP_S = 3.0    # 途切れてから何秒以内なら同一候補とするか
MAX_SPEED = 900    # 許容移動速度(px/秒)。解像度に応じて調整
max_gap = FPS * MAX_GAP_S

# 候補ペア(A→B: AがBより前に終わる・同チーム・近距離)を距離が近い順に貪欲に1対1で結ぶ。
# ※各トラックの前後は最大1本ずつ。これで別々の選手が1人に融合するのを防ぐ
cands = []
for a, sa in summary.items():
    for b, sb in summary.items():
        if a == b or sa["team"] != sb["team"]:
            continue
        gap = sb["f0"] - sa["f1"]
        if gap < 1 or gap > max_gap:
            continue
        d = math.hypot(sa["c1"][0] - sb["c0"][0], sa["c1"][1] - sb["c0"][1])
        if d <= MAX_SPEED * (gap / FPS) + 80:   # gapが長いほど移動を許容
            cands.append((d, a, b))
cands.sort()

parent = {t: t for t in summary}
def find(x):
    r = x
    while parent[r] != r:
        r = parent[r]
    while parent[x] != r:
        parent[x], x = r, parent[x]
    return r

used_succ, used_pred = set(), set()
for d, a, b in cands:
    if a in used_succ or b in used_pred or find(a) == find(b):
        continue
    parent[find(b)] = find(a)
    used_succ.add(a); used_pred.add(b)

groups = defaultdict(list)
for t in summary:
    groups[find(t)].append(t)
print(f"統合前: {len(summary)}トラック → 統合後: {len(groups)}グループ")


In [ ]:
# 合流できなかった短い断片はノイズとして選手一覧から除外し、出場時間順に採番
MIN_PLAYER_S = 3.0   # これ未満のグループは選手扱いしない
def group_sec(mem):
    return sum(summary[m]["n"] for m in mem) / FPS

order = sorted(groups.values(), key=lambda mem: -group_sec(mem))
players = [mem for mem in order if group_sec(mem) >= MIN_PLAYER_S]
print(f"選手候補: {len(players)}人  （{MIN_PLAYER_S}秒未満のノイズ {len(order) - len(players)}件を除外）")

player_of, player_team = {}, {}
for pi, members in enumerate(players, start=1):
    for m in members:
        player_of[m] = pi
    player_team[pi] = Counter(summary[m]["team"] for m in members).most_common(1)[0][0]

with open("players.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["player", "team", "total_seconds", "merged_track_ids"])
    for pi, members in enumerate(players, start=1):
        w.writerow([pi, player_team[pi], round(group_sec(members), 1),
                    " ".join(str(m) for m in sorted(members))])

# detections.csv の track_id を統合後の選手番号に置換（ノイズは除外）
with open("detections.csv") as fin, open("detections_merged.csv", "w", newline="") as fout:
    r = csv.DictReader(fin); w = csv.writer(fout)
    w.writerow(["frame", "time_s", "player", "team", "cx", "foot_y"])
    for row in r:
        pi = player_of.get(int(row["track_id"]))
        if pi is None:
            continue
        w.writerow([row["frame"], row["time_s"], pi, player_team[pi], row["cx"], row["foot_y"]])

print("✅ players.csv / detections_merged.csv を書き出しました\n")
print(f"{'選手':>4} {'team':>4} {'出場秒':>6}  統合したID")
for pi, members in enumerate(players, start=1):
    print(f"{pi:>4} {player_team[pi]:>4} {group_sec(members):>6.1f}  {sorted(members)}")


In [ ]:
# （任意）まとまった選手に実際の背番号を手動で割当（分かる範囲でOK。動画を見て埋める）
PLAYER_NUMBERS = {
    # 選手番号(左) : 背番号(右)   例 →  1: "4",  2: "10",
}
if PLAYER_NUMBERS:
    print("選手番号 → 背番号 の対応:")
    for pi, num in PLAYER_NUMBERS.items():
        print(f"  選手{pi} = 背番号{num}")
else:
    print("必要なら PLAYER_NUMBERS に手動で背番号を入れてください（統合後は約10人なので手作業でも一瞬です）")

from google.colab import files
for f in ["players.csv", "detections_merged.csv"]:
    if Path(f).exists():
        files.download(f)


## 9. シュートチャート（半自動）: どこで打ったか

コート上のどこでシュートを打ったかを、真上から見たコート図にプロットします。
カメラが**固定**なら、コートの基準点4つで一度だけ「**ホモグラフィ変換**」を作れば、
動画上のピクセル位置 → 実際のコート座標に変換できます。

**手順**
1. 下のセルでフレームを表示し、**グリッドの目盛りでピクセル座標を読む**
2. コートの基準点4つ（制限区域=キーの4隅など）のピクセル座標を `SRC_POINTS` に入れる
3. シュートを打った地点（打った選手の足元）のピクセル座標と入/外を `SHOTS` に記録
4. 実行すると `shot_chart.png` が出力される

> 💡 入った/外したは自動判定できないので**あなたが記録**します（動画を見て入=True/外=False）。
> カメラが動く/ズームする映像では単純なホモグラフィはズレます（**固定カメラ前提**）。


In [ ]:
import matplotlib.pyplot as plt

# 座標を読むためのフレームを表示（コートのラインがよく見えるフレーム番号にしてOK）
CAL_FRAME = 0
cap = cv2.VideoCapture(VIDEO_PATH); cap.set(cv2.CAP_PROP_POS_FRAMES, CAL_FRAME)
ok, fr = cap.read(); cap.release()
assert ok, "フレームを読めませんでした。CAL_FRAME を変えてください"

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
ax.set_xticks(range(0, FRAME_W, 100)); ax.set_yticks(range(0, FRAME_H, 100))
ax.grid(color="yellow", alpha=0.4, lw=0.5)
ax.set_title(f"frame {CAL_FRAME} — 黄色い目盛りでピクセル座標(x, y)を読む")
plt.show()


In [ ]:
# コートの4基準点: SRC=上の画像で読んだピクセル座標、DST=実コート座標(m)
# 既定は「キー(制限区域)の4隅」。画像で対応する4点のピクセル座標に置き換えてください。
SRC_POINTS = [
    (0, 0),   # 左手前（ベースライン側・左）
    (0, 0),   # 右手前（ベースライン側・右）
    (0, 0),   # 右奥（フリースローライン側・右）
    (0, 0),   # 左奥（フリースローライン側・左）
]
DST_POINTS = [
    (7.5 - 2.45, 0.0), (7.5 + 2.45, 0.0),
    (7.5 + 2.45, 5.8), (7.5 - 2.45, 5.8),
]

_src = np.array(SRC_POINTS, np.float32)
if np.allclose(_src, 0):
    print("⚠️ SRC_POINTS が未設定です。上の画像でキーの4隅のピクセル座標を読んで入れてください。")
    H = None
else:
    H = cv2.getPerspectiveTransform(_src, np.array(DST_POINTS, np.float32))
    print("✅ ホモグラフィ設定完了")

def to_court(px, py):
    q = cv2.perspectiveTransform(np.array([[[px, py]]], np.float32), H)[0][0]
    return float(q[0]), float(q[1])


In [ ]:
from matplotlib.patches import Rectangle, Circle, Arc

COURT_W, COURT_H = 15.0, 14.0
HOOP = (7.5, 1.575); R3 = 6.75; X_OFF = 0.9

def draw_halfcourt(ax):
    ax.add_patch(Rectangle((0, 0), COURT_W, COURT_H, fill=False, lw=2, color="#444"))
    ax.add_patch(Rectangle((7.5 - 2.45, 0), 4.9, 5.8, fill=False, lw=1.5, color="#444"))  # キー
    ax.add_patch(Arc((7.5, 5.8), 3.6, 3.6, theta1=0, theta2=360, lw=1.5, color="#444"))   # FTサークル
    ax.add_patch(Circle(HOOP, 0.225, fill=False, lw=1.5, color="#c0392b"))                # リング
    ax.plot([7.5 - 0.9, 7.5 + 0.9], [1.2, 1.2], lw=2, color="#c0392b")                    # ボード
    hx, hy = HOOP
    y_corner = hy + np.sqrt(R3**2 - (7.5 - X_OFF)**2)
    ax.plot([X_OFF, X_OFF], [0, y_corner], lw=1.5, color="#444")                          # 3Pコーナー左
    ax.plot([COURT_W - X_OFF, COURT_W - X_OFF], [0, y_corner], lw=1.5, color="#444")      # 3Pコーナー右
    ang = np.linspace(np.arctan2(y_corner - hy, (COURT_W - X_OFF) - hx),
                      np.arctan2(y_corner - hy, X_OFF - hx), 200)
    ax.plot(hx + R3 * np.cos(ang), hy + R3 * np.sin(ang), lw=1.5, color="#444")           # 3Pアーク
    ax.set_xlim(-0.5, COURT_W + 0.5); ax.set_ylim(-0.5, COURT_H + 0.5)
    ax.set_aspect("equal"); ax.axis("off")

# シュート記録: (ピクセルx, ピクセルy, 入ったか, チーム)
#   ピクセル座標 = 打った選手の足元（上のグリッド画像で読む）
#   made: True=入った / False=外した     team: 0 or 1（任意）
SHOTS = [
    # (760, 520, True, 1),
    # (980, 610, False, 0),
]

if H is None:
    print("先にホモグラフィ(SRC_POINTS)を設定してください")
elif not SHOTS:
    print("SHOTS にシュートを記録してください（上のコメントの例を参照）")
else:
    fig, ax = plt.subplots(figsize=(7, 6.6)); draw_halfcourt(ax)
    made_n = miss_n = 0
    for px, py, made, team in SHOTS:
        x, y = to_court(px, py)
        ax.scatter(x, y, s=150, marker="o" if made else "X",
                   facecolor="#2ecc71" if made else "none",
                   edgecolor="#2ecc71" if made else "#e74c3c", linewidths=2, zorder=5)
        made_n += bool(made); miss_n += (not made)
    total = made_n + miss_n
    ax.set_title(f"Shot Chart   made {made_n}/{total}  ({made_n/total:.0%})")
    fig.savefig("shot_chart.png", dpi=120, bbox_inches="tight"); plt.show()
    print(f"✅ shot_chart.png を保存  成功 {made_n} / 失敗 {miss_n}")
    from google.colab import files
    files.download("shot_chart.png")


## 10.（任意）個人の精密追跡（SAM2）

記事後半の「1人の選手を最後まで追う」パートです。**ここは重い＆任意ステップ**です。
SAM2は動画を全フレーム先読みするため長尺だとメモリ落ちしますが、**下のセルが対象選手の周辺15秒だけ自動で切り出して**実行するので安全です。
選手検出＋チーム色分け（1〜6）だけで記事のメイン部分は達成なので、ここは飛ばしてもOKです。


In [ ]:
# 追跡したい選手のトラッキングID（上のチーム分け結果から選ぶ）
TARGET_ID = valid_ids[0] if valid_ids else None
print("追跡対象 ID:", TARGET_ID)

start_frame_idx, start_box = None, None
for (f_idx, x1, y1, x2, y2) in sorted(track_boxes[TARGET_ID]):
    start_frame_idx, start_box = f_idx, (x1, y1, x2, y2); break
print("開始フレーム:", start_frame_idx, "初期bbox:", start_box)


In [ ]:
from ultralytics.models.sam import SAM2VideoPredictor

# ★重要★ SAM2は動画を全フレーム先読みするため、長い動画をそのまま渡すとメモリ落ちします。
# ここでは対象選手が最初に現れる位置から SAM2_SECONDS 秒だけ切り出して実行します（OOM防止）。
SAM2_SECONDS = 15
SAM_VIDEO = "sam_clip.mp4"
ss = start_frame_idx / FPS
!ffmpeg -y -ss {ss} -i "{VIDEO_PATH}" -t {SAM2_SECONDS} -c:v libx264 -an "{SAM_VIDEO}" -loglevel error
print(f"✂️ SAM2用に {ss:.1f}s から {SAM2_SECONDS}s を切出し: {SAM_VIDEO}")

# 軽量設定: tinyモデル + imgsz=512。精度重視なら model="sam2.1_b.pt", imgsz=1024
overrides = dict(conf=0.25, task="segment", mode="predict",
                 imgsz=512, model="sam2.1_t.pt", device=DEVICE, verbose=False)
predictor = SAM2VideoPredictor(overrides=overrides)

x1, y1, x2, y2 = start_box
sam_results = predictor(source=SAM_VIDEO, bboxes=[[x1, y1, x2, y2]], labels=[1])
print("✅ SAM2 追跡完了。フレーム数:", len(sam_results))


In [ ]:
sam_out = "output_sam2_track.mp4"
writer2 = cv2.VideoWriter(sam_out, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (FRAME_W, FRAME_H))
overlay_color = np.array([0, 255, 255], dtype=np.uint8)  # 黄色
for res in sam_results:
    base = res.orig_img.copy()
    if res.masks is not None and len(res.masks) > 0:
        m = res.masks.data[0].cpu().numpy().astype(np.uint8)
        if m.shape != base.shape[:2]:
            m = cv2.resize(m, (base.shape[1], base.shape[0]))
        mb = m > 0
        base[mb] = (0.5 * base[mb] + 0.5 * overlay_color).astype(np.uint8)
    writer2.write(base)
writer2.release()
print("✅ 書き出し完了:", sam_out)


In [ ]:
!ffmpeg -y -i output_sam2_track.mp4 -vcodec libx264 -movflags +faststart preview_sam2.mp4 -loglevel error
from IPython.display import HTML
from base64 import b64encode
data_url = "data:video/mp4;base64," + b64encode(open("preview_sam2.mp4", "rb").read()).decode()
HTML(f'<video width="640" controls><source src="{data_url}" type="video/mp4"></video>')


---
## 📝 チューニング＆ロードマップ

**チーム分けが不安定なとき**
- セクション4の `MIN_HEIGHT_RATIO` / `MIN_MOVEMENT` を上げてベンチ・観客をさらに除外
- 確認用セルの表を見て、本来の選手が `×` なら各しきい値を下げる
- ユニフォームが白×薄色で近い場合は `torso_lab` の切り出し範囲を調整

**処理が重い/長い試合**
- 検出モデルを `yolo11n.pt` に（速い）。まずは `TRIM_SECONDS` で短く動作確認

**このノートブックの到達点と、この先（個人成績・戦術分析へ）**

| 段階 | 内容 | 状態 |
|---|---|---|
| ① 逐次処理で長い動画対応 | OOMせず1試合分を処理 | ✅ 本ノート |
| ② チーム分け精度 | LAB色＋ベンチ除外 | ✅ 本ノート |
| ③ 位置データCSV | `detections.csv`（フレーム毎の選手位置） | ✅ 本ノート |
| ⑤ 背番号OCR（検証） | トラックID→実際の選手番号・読み取り成功率 | ✅ 本ノート(§7) |
| ⑤' トラック統合(re-ID) | 分裂IDを動きの連続性で1人に結合（背番号に依存しない） | ✅ 本ノート(§8) |
| ④ コートのホモグラフィ | カメラ視点→実寸コート座標 | ✅ 本ノート(§9) |
| ⑥ シュートチャート | どこで打ったか（半自動：入外は手動記録） | ✅ 本ノート(§9) |
| ⑥' 個人成績 | 走行距離・出場時間・ヒートマップ | ⏭ ④の座標を使えば追加可 |
| ⑦ 半自動プレー分析 | 種別をマーク→成功/失敗を判定→成功率 | ⏭ 研究的・最後 |

コストは Colab 無料枠 + OSSモデルのみ = **0円** で動きます。
